In [1]:
import os, sys, torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

REPO = "arslanmasood/BioXMol"

# 1. download the code + weights from this repo
for f in ["modeling_bioxmol.py", "featurizer.py"]:
    code_dir = os.path.dirname(hf_hub_download(REPO, f))
sys.path.insert(0, code_dir)
from modeling_bioxmol import GatedGraphNeuralNetwork
from featurizer import smiles2graph

# 2. load the encoder
model = GatedGraphNeuralNetwork(n_edge=1, in_dim=75, n_conv=6, fc_dims=[1024, 128])
model.load_state_dict(load_file(hf_hub_download(REPO, "bioxmol_soft_seed0.safetensors")))
model.eval()

# 3. encode a list of SMILES -> embeddings

def encode(smiles_list, layer="first_fc"):
    graphs = [smiles2graph(s) for s in smiles_list]
    n = max(nf.shape[0] for _, nf in graphs)            # pad to the largest molecule
    A, F, M = [], [], []
    for adj, nf in graphs:
        adj = torch.as_tensor(adj, dtype=torch.float)
        nf = torch.as_tensor(nf, dtype=torch.float)
        a = torch.zeros(n, n);           a[:adj.shape[0], :adj.shape[1]] = adj
        f = torch.zeros(n, nf.shape[1]); f[:nf.shape[0]] = nf
        m = torch.zeros(n, 1);           m[:nf.shape[0]] = 1.0   # 1 = real atom
        A.append(a); F.append(f); M.append(m)
    return model.embed(torch.stack(A), torch.stack(F), torch.stack(M), layer=layer)

# example
emb = encode(["CCO", "CC(=O)Oc1ccccc1C(=O)O"])
print(emb.shape)   # torch.Size([2, 1024])

torch.Size([2, 1024])


In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm

CSV_PATH = "/scratch/cds/MolML/JEPA/data/merged_data_deduplicated_with_benchmarks_log.csv"
OUTPUT_DIR = "/scratch/cds/MolML/JEPA/bioxmol"
BATCH_SIZE = 256
USE_CACHE = False

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH, low_memory=False).iloc[4323582:]
print(f"Loaded {len(df)} rows.")

failed_count = 0
for batch_start in tqdm(range(0, len(df), BATCH_SIZE), desc="Computing BioXMol embeddings"):
    batch_end = min(batch_start + BATCH_SIZE, len(df))
    batch_indices = df.index[batch_start:batch_end].tolist()

    if USE_CACHE:
        batch_indices = [
            idx
            for idx in batch_indices
            if not os.path.exists(os.path.join(OUTPUT_DIR, f"bioxmol_{idx}.npy"))
        ]
    if not batch_indices:
        continue

    valid_smiles = []
    valid_indices = []
    for idx in batch_indices:
        smiles = df.at[idx, "smiles"]
        if not isinstance(smiles, str) or not smiles:
            failed_count += 1
            continue
        try:
            smiles2graph(smiles)
        except Exception:
            failed_count += 1
            continue
        valid_smiles.append(smiles)
        valid_indices.append(idx)

    if not valid_smiles:
        continue

    with torch.no_grad():
        embeddings = encode(valid_smiles).cpu().numpy()

    for embedding, idx in zip(embeddings, valid_indices):
        out_path = os.path.join(OUTPUT_DIR, f"bioxmol_{idx}.npy")
        np.save(out_path, embedding)

print(f"Done. Total failures: {failed_count}")

Loaded 365254 rows.


Computing BioXMol embeddings: 100%|██████████| 1427/1427 [56:41<00:00,  2.38s/it] 

Done. Total failures: 0
